# Step 6 — NammaMart ELT pipeline test

This notebook runs the ELT implementation locally. The raw layer is loaded from the CSV files as text; staging and mart transformations are executed from the checked-in SQL files.

Flow: load raw → transform staging with SQL → build marts with SQL → inspect row counts and business outputs.

In [ ]:
from pathlib import Path
import sys
import duckdb

# Import the same ELT module used by the Airflow DAG.
sys.path.insert(0, str(Path('include').resolve()))
from nammamart_elt import run_elt

# Keep the ELT warehouse separate from the ETL warehouse for comparison.
warehouse = Path('assignment/warehouse_samy_elt_step6.duckdb')
metrics = run_elt(
    warehouse_path=warehouse,
    run_id='step6-local-test',
)
for name, value in metrics.items():
    print(f'{name}: {value:,}' if isinstance(value, int) else f'{name}: {value}')

In [ ]:
# Inspect every layer to prove that raw data, staging decisions and marts exist.
con = duckdb.connect(str(warehouse), read_only=True)
display(con.execute("""
    SELECT table_name, estimated_size
    FROM duckdb_tables()
    WHERE schema_name = 'main'
    ORDER BY table_name
""").fetchdf())

# Raw rows are retained while staging exposes pass/fail decisions.
display(con.execute("SELECT dq_status, count(*) AS rows FROM stg_orders GROUP BY dq_status").fetchdf())
display(con.execute("SELECT * FROM mart_category_margin ORDER BY category").fetchdf())
display(con.execute("SELECT * FROM mart_daily_store_kpis ORDER BY order_date, zone, store_id LIMIT 10").fetchdf())
display(con.execute("SELECT * FROM mart_top_customers LIMIT 10").fetchdf())
con.close()

In [ ]:
# Re-running the initial ELT load replaces the derived tables, so this test
# confirms the initial pipeline does not duplicate rows on rerun.
second_metrics = run_elt(warehouse_path=warehouse, run_id='step6-local-rerun')
con = duckdb.connect(str(warehouse), read_only=True)
raw_count = con.execute('SELECT count(*) FROM raw_orders').fetchone()[0]
stg_count = con.execute('SELECT count(*) FROM stg_orders').fetchone()[0]
con.close()
print(f'Raw orders after rerun: {raw_count:,}')
print(f'Staging orders after rerun: {stg_count:,}')
print('The initial ELT run is deterministic; incremental append/merge is implemented in the next step.')